# preprocess_bayesian.py
Builds the Bayesian regression modeling table from the preprocessed housing panel.

In [ ]:
"""Build the Bayesian regression modeling table from the preprocessed housing panel.

Input: outputs/boston_housing_preprocessed.csv
Output: outputs/bayesian_data.csv
"""
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "outputs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
INPUT_FILE  = PROJECT_ROOT / "outputs" / "boston_housing_preprocessed.csv"
OUTPUT_FILE = PROJECT_ROOT / "outputs" / "bayesian_data.csv"
START_DATE  = "2018-01-31"
END_DATE    = "2024-12-31"
COLS = [
    "zip_code", "date", "income_needed", "MORTGAGE30US",
    "UNRATE", "CPIAUCSL", "zhvi_yoy_pct", "inventory",
    "days_to_pending", "location_tier"
]

In [ ]:
def main():
    print("Preparing Bayesian regression dataset\n")

    if not INPUT_FILE.exists():
        print(f"[ERROR] {INPUT_FILE} not found. Run preprocess.py first.")
        return

    df = pd.read_csv(INPUT_FILE, dtype={"zip_code": str})
    df["date"] = pd.to_datetime(df["date"])

    df = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)].copy()
    print(f"After date filter ({START_DATE} to {END_DATE}): {len(df):,} rows")

    df = df[COLS].copy()

    before = len(df)
    df = df.dropna()
    dropped = before - len(df)
    print(f"Dropped {dropped:,} rows with nulls")

    threshold = df["income_needed"].quantile(0.75)
    df["high_stress"] = (df["income_needed"] > threshold).astype(int)
    print(f"high_stress threshold (75th percentile): ${threshold:,.2f}")

    df = df.sort_values(["zip_code", "date"]).reset_index(drop=True)

    OUTPUT_FILE.parent.mkdir(exist_ok=True)
    df.to_csv(OUTPUT_FILE, index=False)

    print(f"\nSaved {OUTPUT_FILE}")
    print(f"{len(df):,} rows | {df['zip_code'].nunique()} ZIP codes")
    print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
    print(f"Nulls remaining: {df.isnull().sum().sum()}")
    print(f"high_stress distribution:\n{df['high_stress'].value_counts()}")

    print("\nLocation tier breakdown:")
    tier_counts = df.drop_duplicates("zip_code")["location_tier"].value_counts()
    for tier, count in tier_counts.items():
        print(f"  {tier}: {count} ZIPs")

In [ ]:
if __name__ == "__main__":
    main()